# 05 Sequence Dataset

**Phase 4 — Sequential Learning Analytics**  
Research contract: `PHASE4_RESEARCH_CONTRACT_v1.md`  
Schema version: `seq_v1`

This notebook converts flat snapshot CSVs into ordered learning sequences
suitable for LSTM, GRU, and TAG construction.  
It does **not** connect to Supabase. All inputs are offline CSV snapshots.

## Pipeline

```
Raw snapshot CSVs
  sequence_<date>_<batch>.csv   ← vw_dataset_sequence_level
  attempt_<date>_<batch>.csv    ← vw_dataset_attempt_level
  outcome_<date>_<batch>.csv    ← computed 2C3L labels
        │
        ▼
  [1] Load + validate snapshots
  [2] Canonicalize events (deduplicate client/server pairs)
  [3] Compute cutoff_timestamp per learner
  [4] Anti-leakage check — no post-cutoff features
  [5] Build per-step feature vectors
  [6] Validate labels (label_source / label_validity)
  [7] Student-level split (GroupShuffleSplit, frozen ledger)
  [8] Pad + mask sequences → tensors
  [9] Fit vocabulary + scaler on train split only
 [10] Write artifacts + sequence manifest
 [11] Validation summary
```

> ⚠️ **TECHNICAL VALIDATION ONLY** — generated from a ≤10-student mock dataset.  
> Not suitable for research conclusions. Final results require ≥60 participants.

## 0. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)

# ── Paths ─────────────────────────────────────────────────────────────────────
RAW_DIR  = Path('data/raw')
SEQ_DIR  = Path('data/sequences')
SEQ_DIR.mkdir(parents=True, exist_ok=True)

# Set to specific filenames to pin a run; None = auto-select newest
SEQUENCE_CSV : str | None = None
ATTEMPT_CSV  : str | None = None
OUTCOME_CSV  : str | None = None

# ── Research contract parameters (PHASE4_RESEARCH_CONTRACT_v1.md) ─────────────
SCHEMA_VERSION          = 'seq_v1'
PHASE3_SOURCE_SHA       = '193b18949e40e6bd3bbfb70034a5772ce51d1b7e'
RANDOM_STATE            = 42
TEST_SIZE               = 0.20
AT_RISK_THRESHOLD       = 65.0   # total_2c3l_score < 65 → at_risk=1
DEDUP_WINDOW_SEC        = 5      # duplicate-pair detection window
DUPLICATE_EVENT_TYPES   = {'sql_run', 'submit_answer'}  # both fired by client+server
MAX_SEQ_LEN_PERCENTILE  = 95     # cap sequence length at this percentile
VALID_LABEL_SOURCES     = {'teacher_reviewed', 'expert_validated', 'auto_scored_validated'}
PILOT_LABEL_SOURCES     = {'auto_generated'}  # allowed for pipeline validation, not thesis

# ── Canonical 2C3L criterion keys (frozen — Draft-06) ─────────────────────────
CANONICAL_CRITERIA = [
    'c1_correctness_result',
    'c2_semantic_consistency',
    'l1_logical_reasoning',
    'l2_learning_process',
    'l3_difficulty_complexity',
]

# ── Anti-leakage column blacklist (§10 of research contract) ──────────────────
LEAKAGE_COLS = (
    set(CANONICAL_CRITERIA)
    | {f'{k}_score' for k in CANONICAL_CRITERIA}
    | {f'{k}_max'   for k in CANONICAL_CRITERIA}
    | {'total_2c3l_score', 'at_risk', 'is_correct_final', 'score_final',
       'rubric_applied_version', 'grade_letter'}
)

# ── Block event types (reserved for future phases — not yet collected) ─────────
BLOCK_EVENT_TYPES = {'block_add', 'block_move', 'block_delete', 'block_submit'}

print(f'Schema version : {SCHEMA_VERSION}')
print(f'Phase 3 source : {PHASE3_SOURCE_SHA}')
print(f'AT_RISK_THRESHOLD : {AT_RISK_THRESHOLD} (canonical 2C3L)')
print(f'Random state   : {RANDOM_STATE}')

## 1. Load and validate snapshots

In [ ]:
def newest_matching(pattern: str) -> Path | None:
    files = sorted(RAW_DIR.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0] if files else None

seq_path     = Path(SEQUENCE_CSV) if SEQUENCE_CSV else newest_matching('sequence_*.csv')
attempt_path = Path(ATTEMPT_CSV)  if ATTEMPT_CSV  else newest_matching('attempt_*.csv')
outcome_path = Path(OUTCOME_CSV)  if OUTCOME_CSV  else newest_matching('outcome_*.csv')

for label, p in [('sequence', seq_path), ('attempt', attempt_path), ('outcome', outcome_path)]:
    if p is None:
        raise FileNotFoundError(
            f'No {label} CSV found in {RAW_DIR}. '
            f'Run the mock pipeline process step or export {label} data first.'
        )
    print(f'{label:10s}: {p}')

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    h.update(p.read_bytes())
    return h.hexdigest()[:16]

seq_df     = pd.read_csv(seq_path,     encoding='utf-8-sig')
attempt_df = pd.read_csv(attempt_path, encoding='utf-8-sig')
outcome_df = pd.read_csv(outcome_path, encoding='utf-8-sig')

print(f'\nsequence rows : {len(seq_df):,}  |  columns : {seq_df.shape[1]}')
print(f'attempt rows  : {len(attempt_df):,}  |  columns : {attempt_df.shape[1]}')
print(f'outcome rows  : {len(outcome_df):,}  |  columns : {outcome_df.shape[1]}')

In [ ]:
# Required columns for each snapshot
REQ_SEQ = {
    'academy_member_id', 'batch_code', 'task_code',
    'session_id', 'event_order', 'event_type', 'event_time',
}
REQ_ATTEMPT = {
    'academy_member_id', 'batch_code', 'task_code',
    'attempt_no', 'is_correct', 'created_at',
}
REQ_OUTCOME = {
    'participant_code', 'batch_code', 'task_code',
    'submission_id', 'submitted_at',
    'total_2c3l_score', 'at_risk', 'label_source', 'label_validity',
}

def check_required(df: pd.DataFrame, required: set, name: str) -> None:
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f'{name} missing required columns: {missing}')
    print(f'  {name:12s} required columns: OK')

check_required(seq_df,     REQ_SEQ,     'sequence')
check_required(attempt_df, REQ_ATTEMPT, 'attempt')
check_required(outcome_df, REQ_OUTCOME, 'outcome')

# Normalise: outcome uses participant_code; rename to match other tables
if 'participant_code' in outcome_df.columns and 'academy_member_id' not in outcome_df.columns:
    outcome_df = outcome_df.rename(columns={'participant_code': 'academy_member_id'})

# Parse timestamps
seq_df['event_time'] = pd.to_datetime(seq_df['event_time'], utc=True, errors='coerce')
attempt_df['created_at'] = pd.to_datetime(attempt_df['created_at'], utc=True, errors='coerce')
if 'submitted_at' in outcome_df.columns:
    outcome_df['submitted_at'] = pd.to_datetime(outcome_df['submitted_at'], utc=True, errors='coerce')

print('\nTimestamp parsing: OK')

## 2. Canonicalize events — deduplicate client/server pairs

Both `sql_run` and `submit_answer` are fired by both the client page (optimistic)
and the server route handler, producing two consecutive identical events per action.

**Rule (research contract §8):** For each duplicate pair, retain the event with the
**higher `event_order`** (server-side) and drop the lower (client-side).

In [ ]:
seq_df = seq_df.sort_values(
    ['session_id', 'event_order'],
    ascending=[True, True]
).reset_index(drop=True)

def mark_client_duplicates(df: pd.DataFrame, dup_types: set, window_sec: int) -> pd.Series:
    """
    Returns a boolean Series where True = row is the client-side duplicate to drop.
    Detects consecutive same-type events in the same session within window_sec.
    """
    is_dup = pd.Series(False, index=df.index)
    mask   = df['event_type'].isin(dup_types)
    cands  = df[mask].copy()

    if cands.empty:
        return is_dup

    cands['prev_session']    = cands.groupby('session_id')['session_id'].shift(1)
    cands['prev_event_type'] = cands.groupby('session_id')['event_type'].shift(1)
    cands['prev_order']      = cands.groupby('session_id')['event_order'].shift(1)
    cands['prev_time']       = cands.groupby('session_id')['event_time'].shift(1)
    cands['delta_order']     = cands['event_order'] - cands['prev_order']
    cands['delta_sec']       = (
        cands['event_time'] - cands['prev_time']
    ).dt.total_seconds().abs()

    pair_mask = (
        (cands['prev_session'] == cands['session_id']) &
        (cands['prev_event_type'] == cands['event_type']) &
        (cands['delta_order'] == 1) &
        (cands['delta_sec'] <= window_sec)
    )
    # The LOWER event_order in the pair is the client-side one — mark its predecessor
    prev_indices = cands.index[pair_mask] - 1
    valid_prev   = prev_indices[prev_indices >= 0]
    is_dup.iloc[valid_prev] = True
    return is_dup

dup_mask = mark_client_duplicates(seq_df, DUPLICATE_EVENT_TYPES, DEDUP_WINDOW_SEC)

seq_df['dropped_as_duplicate'] = dup_mask
canonical_df = seq_df[~dup_mask].copy()

n_dropped = dup_mask.sum()
print(f'Raw events     : {len(seq_df):,}')
print(f'Dropped (dups) : {n_dropped:,}  (window={DEDUP_WINDOW_SEC}s)')
print(f'Canonical      : {len(canonical_df):,}')

# Block events should not be present in Phase 4 data
block_found = canonical_df['event_type'].isin(BLOCK_EVENT_TYPES).sum()
print(f'Block events   : {block_found}  (expected 0 — BlockSqlBuilder not connected in Phase 4)')

## 3. Compute cutoff_timestamp per learner × task

**Cutoff rule (research contract §3):** The cutoff is the timestamp of the learner's
first `submit_answer` event on the task. No feature derived from events at or after
the cutoff may enter model inputs.

In [ ]:
submit_events = canonical_df[
    canonical_df['event_type'] == 'submit_answer'
].copy()

# First submission per learner × task
cutoff_df = (
    submit_events
    .sort_values('event_time')
    .groupby(['academy_member_id', 'task_code'], as_index=False)
    .first()[['academy_member_id', 'task_code', 'event_time']]
    .rename(columns={'event_time': 'cutoff_timestamp'})
)

print(f'Learner × task cutoffs computed: {len(cutoff_df)}')
print(cutoff_df.head())

# Merge cutoff back into canonical events
canonical_df = canonical_df.merge(
    cutoff_df, on=['academy_member_id', 'task_code'], how='left'
)

# Flag events at or after cutoff — these must not contribute to model inputs
canonical_df['is_post_cutoff'] = (
    canonical_df['event_time'] >= canonical_df['cutoff_timestamp']
).where(canonical_df['cutoff_timestamp'].notna(), other=False)

pre_cutoff_df = canonical_df[~canonical_df['is_post_cutoff']].copy()

print(f'\nPre-cutoff events  : {len(pre_cutoff_df):,}')
print(f'Post-cutoff events : {canonical_df["is_post_cutoff"].sum():,}  (excluded from model inputs)')

## 4. Anti-leakage check

Assert that no column in the pre-cutoff feature set contains a blacklisted label
or post-submission score. See research contract §10.

In [ ]:
feature_cols = set(pre_cutoff_df.columns)
leaked = sorted(feature_cols & LEAKAGE_COLS)

if leaked:
    raise ValueError(
        f'LEAKAGE DETECTED — the following columns must not appear in model inputs: {leaked}'
    )
print('Anti-leakage check: PASS — no blacklisted columns in pre-cutoff event data')

## 5. Build per-step feature vectors

Each event becomes one timestep in the learner's sequence.  
Features are derived **only** from pre-cutoff events and attempt metadata.

Per-step features (10 dimensions, extensible):

| # | Feature | Source |
|---|---------|--------|
| 0 | `event_type_code` | vocabulary integer |
| 1 | `duration_from_start_norm` | normalized session-relative time |
| 2 | `is_sql_run` | binary |
| 3 | `is_submit` | binary |
| 4 | `is_error_event` | binary |
| 5 | `cumulative_run_count` | running count up to this step |
| 6 | `cumulative_submit_count` | running count |
| 7 | `cumulative_error_count` | running count |
| 8 | `attempt_is_correct` | from attempt table (0/1/NaN) |
| 9 | `step_position_norm` | position / max_seq_len |

In [ ]:
# Build event type vocabulary (fit on ALL data — vocabulary is not a model parameter)
all_types = sorted(canonical_df['event_type'].dropna().unique().tolist())
# Reserve slots for block events even though they carry no data in Phase 4
reserved   = sorted(BLOCK_EVENT_TYPES - set(all_types))
vocab_list = all_types + reserved
vocab      = {et: i + 1 for i, et in enumerate(vocab_list)}  # 0 = padding

print(f'Vocabulary size : {len(vocab)} event types  (0 reserved for padding)')
for et, code in vocab.items():
    note = ' ← reserved (block, not collected)' if et in BLOCK_EVENT_TYPES else ''
    print(f'  {code:3d}  {et}{note}')

In [ ]:
# Join attempt correctness onto events where available
# Match on: academy_member_id × task_code × attempt_no (attempt events carry attempt_no in event_value)
attempt_df['attempt_no'] = pd.to_numeric(attempt_df['attempt_no'], errors='coerce')
attempt_correct = attempt_df[['academy_member_id', 'task_code', 'attempt_no', 'is_correct']].copy()
attempt_correct['is_correct'] = attempt_correct['is_correct'].astype(float)

# Build per-step features for the pre-cutoff stream
records = []
for (learner_id, task_code), grp in pre_cutoff_df.groupby(
    ['academy_member_id', 'task_code'], sort=False
):
    grp = grp.sort_values('event_order').reset_index(drop=True)
    dur_max = grp['duration_from_start'].replace('', np.nan).astype(float).max()
    dur_max = dur_max if (dur_max and dur_max > 0) else 1.0

    cum_run    = 0
    cum_submit = 0
    cum_error  = 0

    for step_i, row in grp.iterrows():
        et   = row['event_type']
        dur  = float(row['duration_from_start']) if row['duration_from_start'] not in ('', None) and pd.notna(row['duration_from_start']) else 0.0

        is_run    = int(et == 'sql_run')
        is_sub    = int(et == 'submit_answer')
        is_err    = int(et in {'sql_error'})

        cum_run    += is_run
        cum_submit += is_sub
        cum_error  += is_err

        records.append({
            'academy_member_id':       learner_id,
            'task_code':               task_code,
            'event_id':                row.get('event_id', ''),
            'event_order':             int(row['event_order']),
            'event_type':              et,
            'event_time':              row['event_time'],
            'event_type_code':         vocab.get(et, 0),
            'duration_from_start_norm': round(dur / dur_max, 6),
            'is_sql_run':              is_run,
            'is_submit':               is_sub,
            'is_error_event':          is_err,
            'cumulative_run_count':    cum_run,
            'cumulative_submit_count': cum_submit,
            'cumulative_error_count':  cum_error,
            'attempt_is_correct':      np.nan,  # filled below
            'step_position_norm':      0.0,     # filled after max_len is known
        })

feature_df = pd.DataFrame(records)
print(f'Feature rows (pre-cutoff steps): {len(feature_df):,}')
print(feature_df.dtypes)

In [ ]:
# Fill attempt_is_correct from attempt table where event_type matches
# Use cumulative_submit_count as proxy for attempt_no on submit events
for (lid, tc), grp in feature_df.groupby(['academy_member_id', 'task_code']):
    sub_mask = grp['event_type'] == 'submit_answer'
    sub_idx  = grp.index[sub_mask]
    for rank, idx in enumerate(sub_idx, start=1):
        match = attempt_correct[
            (attempt_correct['academy_member_id'] == lid) &
            (attempt_correct['task_code'] == tc) &
            (attempt_correct['attempt_no'] == rank)
        ]
        if not match.empty:
            feature_df.at[idx, 'attempt_is_correct'] = float(match.iloc[0]['is_correct'])

# Sequence length per learner × task
seq_lengths = feature_df.groupby(['academy_member_id', 'task_code']).size()
max_len_raw = int(np.percentile(seq_lengths.values, MAX_SEQ_LEN_PERCENTILE))
max_len_raw = max(max_len_raw, 1)

# Fill step_position_norm
for (lid, tc), grp in feature_df.groupby(['academy_member_id', 'task_code']):
    n = len(grp)
    feature_df.loc[grp.index, 'step_position_norm'] = [
        round(i / max(n - 1, 1), 6) for i in range(n)
    ]

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(seq_lengths.values, bins=20, edgecolor='black')
ax.axvline(max_len_raw, color='red', linestyle='--', label=f'P{MAX_SEQ_LEN_PERCENTILE}={max_len_raw}')
ax.set_xlabel('Sequence length (steps)')
ax.set_ylabel('Count')
ax.set_title('Sequence length distribution (learner × task)')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/seq_length_dist.png', dpi=150)
plt.show()

print(f'Sequence lengths — min:{seq_lengths.min()}  median:{seq_lengths.median():.0f}  '
      f'P{MAX_SEQ_LEN_PERCENTILE}:{max_len_raw}  max:{seq_lengths.max()}')

## 6. Validate labels — apply label priority rules

Research contract §5: only `teacher_reviewed`, `expert_validated`, or
`auto_scored_validated` labels may be used for thesis conclusions.
`auto_generated` labels are marked pilot-only.

In [ ]:
# Outcome table: one row per learner × task submission
# Derive a single canonical label per learner (last task submitted, or aggregate)
# For Phase 4 pilot: use the outcome with the lowest (worst) 2C3l score per learner,
# i.e., if any task is at_risk=1 the learner is at_risk.

outcome_df['total_2c3l_score'] = pd.to_numeric(outcome_df['total_2c3l_score'], errors='coerce')
outcome_df['at_risk'] = pd.to_numeric(outcome_df['at_risk'], errors='coerce')

label_counts = outcome_df['label_source'].value_counts()
print('Label source distribution:')
print(label_counts.to_string())
print()

# Learner-level label: worst outcome across all tasks
learner_labels = (
    outcome_df
    .sort_values('total_2c3l_score', ascending=True)  # worst first
    .groupby('academy_member_id', as_index=False)
    .first()[['academy_member_id', 'at_risk', 'label_source', 'label_validity',
               'total_2c3l_score', 'is_teacher_reviewed']]
)

# Classify for thesis eligibility
learner_labels['thesis_eligible'] = learner_labels['label_source'].isin(VALID_LABEL_SOURCES)
learner_labels['pilot_only']      = learner_labels['label_source'].isin(PILOT_LABEL_SOURCES)

print(f'Total learners in outcome table : {len(learner_labels)}')
print(f'  thesis_eligible  : {learner_labels["thesis_eligible"].sum()}')
print(f'  pilot_only       : {learner_labels["pilot_only"].sum()}')
print(f'  at_risk=1        : {(learner_labels["at_risk"] == 1).sum()}')
print(f'  at_risk=0        : {(learner_labels["at_risk"] == 0).sum()}')
print(f'  no label (NaN)   : {learner_labels["at_risk"].isna().sum()}')

if learner_labels['thesis_eligible'].sum() == 0:
    print('\n⚠️  WARNING: No thesis-eligible labels exist in this dataset.')
    print('   All labels are auto_generated / pilot_only.')
    print('   This dataset is valid for TECHNICAL PIPELINE VALIDATION ONLY.')

## 7. Student-level split — GroupShuffleSplit, frozen ledger

Split is applied at the **learner** level. No learner may appear in both train and test.
The resulting `split_assignments.parquet` is the authoritative ledger for all downstream
notebooks (LSTM, GRU, comparison).

In [ ]:
# Only learners with a valid label (or pilot label) may enter the split
eligible = learner_labels[
    learner_labels['at_risk'].notna()
].copy().reset_index(drop=True)

print(f'Eligible learners for split: {len(eligible)}')

if len(eligible) < 3:
    raise ValueError(
        f'Only {len(eligible)} eligible learner(s). '
        'GroupShuffleSplit requires at least 2 groups in each split. '
        'Collect more data before running the sequence pipeline.'
    )

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
X_dummy = np.zeros((len(eligible), 1))
y_dummy = eligible['at_risk'].values
groups  = eligible['academy_member_id'].values

train_idx, test_idx = next(gss.split(X_dummy, y_dummy, groups))

eligible['split'] = 'train'
eligible.loc[test_idx, 'split'] = 'test'

train_learners = set(eligible.loc[train_idx, 'academy_member_id'])
test_learners  = set(eligible.loc[test_idx,  'academy_member_id'])
overlap        = train_learners & test_learners
assert not overlap, f'SPLIT LEAKAGE: learners appear in both splits: {overlap}'

print(f'Train learners : {len(train_learners)}')
print(f'Test  learners : {len(test_learners)}')
print(f'Overlap        : {len(overlap)}  (must be 0)')
print(f'At_risk in train: {eligible[eligible["split"]=="train"]["at_risk"].value_counts().to_dict()}')
print(f'At_risk in test : {eligible[eligible["split"]=="test" ]["at_risk"].value_counts().to_dict()}')

In [ ]:
# Save the frozen split ledger — all downstream notebooks load this file
split_ledger = eligible[[
    'academy_member_id', 'split', 'at_risk',
    'label_source', 'label_validity', 'thesis_eligible',
    'total_2c3l_score', 'is_teacher_reviewed',
]].copy()

ledger_path = SEQ_DIR / 'split_assignments.parquet'
split_ledger.to_parquet(ledger_path, index=False)
print(f'Split ledger saved: {ledger_path}  ({len(split_ledger)} rows)')
display(split_ledger)

## 8. Build sequence index and canonical events artifact

In [ ]:
# Canonical events (raw + dedup flag + cutoff flag + split assignment)
canonical_export = seq_df[[
    'academy_member_id', 'batch_code', 'task_code',
    'session_id', 'event_id', 'event_order',
    'event_type', 'event_value', 'duration_from_start',
    'event_time', 'dropped_as_duplicate',
]].copy()

# Merge cutoff and post-cutoff flag from canonical_df
canonical_export = canonical_export.merge(
    canonical_df[['event_id', 'cutoff_timestamp', 'is_post_cutoff']].drop_duplicates('event_id'),
    on='event_id', how='left',
)

# Merge split assignment
canonical_export = canonical_export.merge(
    split_ledger[['academy_member_id', 'split']],
    on='academy_member_id', how='left',
)

canonical_path = SEQ_DIR / 'canonical_events.parquet'
canonical_export.to_parquet(canonical_path, index=False)
print(f'Canonical events saved: {canonical_path}  ({len(canonical_export)} rows)')

# Sequence index — one row per learner × task
seq_index = (
    feature_df
    .groupby(['academy_member_id', 'task_code'], as_index=False)
    .agg(n_steps=('event_order', 'count'),
         first_event_time=('event_time', 'min'),
         last_event_time=('event_time', 'max'))
)
seq_index = seq_index.merge(
    cutoff_df, on=['academy_member_id', 'task_code'], how='left'
)
seq_index = seq_index.merge(
    split_ledger[['academy_member_id', 'split', 'at_risk', 'label_source', 'label_validity']],
    on='academy_member_id', how='left',
)

seq_index_path = SEQ_DIR / 'sequence_index.parquet'
seq_index.to_parquet(seq_index_path, index=False)
print(f'Sequence index saved: {seq_index_path}  ({len(seq_index)} rows)')
display(seq_index.head())

## 9. Pad + mask sequences → tensors

Vocabulary and scaler are fit **on training data only**, then applied to test.

In [ ]:
NUMERIC_FEATURES = [
    'event_type_code',
    'duration_from_start_norm',
    'is_sql_run',
    'is_submit',
    'is_error_event',
    'cumulative_run_count',
    'cumulative_submit_count',
    'cumulative_error_count',
    'attempt_is_correct',
    'step_position_norm',
]
N_FEATURES = len(NUMERIC_FEATURES)

# Assign learners to splits
split_map = dict(zip(split_ledger['academy_member_id'], split_ledger['split']))
label_map = dict(zip(split_ledger['academy_member_id'], split_ledger['at_risk']))
feature_df['split'] = feature_df['academy_member_id'].map(split_map)

learner_task_ids = (
    feature_df[feature_df['split'].notna()]
    .groupby(['academy_member_id', 'task_code'])
    .size()
    .reset_index()[['academy_member_id', 'task_code']]
    .values.tolist()
)

# Fit scaler on train sequences only
train_steps = feature_df[feature_df['split'] == 'train'][NUMERIC_FEATURES].copy()
train_steps['attempt_is_correct'] = train_steps['attempt_is_correct'].fillna(0.0)

scaler = StandardScaler()
scaler.fit(train_steps)

print(f'Scaler fit on {len(train_steps)} training steps')
print(f'Max sequence length (P{MAX_SEQ_LEN_PERCENTILE}): {max_len_raw} steps')
print(f'Feature dimensions: {N_FEATURES}')

In [ ]:
def build_tensors(split_name: str) -> tuple:
    """Return (X, y, mask, learner_ids) for the given split."""
    learners = [lid for lid, sp in split_map.items() if sp == split_name]
    # Include all task sequences for each learner — one sequence per (learner × task)
    pairs = [
        (lid, tc)
        for lid, tc in learner_task_ids
        if lid in set(learners)
    ]

    X    = np.zeros((len(pairs), max_len_raw, N_FEATURES), dtype=np.float32)
    mask = np.zeros((len(pairs), max_len_raw),             dtype=bool)
    y    = np.zeros(len(pairs),                            dtype=np.int8)
    ids  = []

    for i, (lid, tc) in enumerate(pairs):
        grp = feature_df[
            (feature_df['academy_member_id'] == lid) &
            (feature_df['task_code'] == tc)
        ].sort_values('event_order')[NUMERIC_FEATURES].copy()

        grp['attempt_is_correct'] = grp['attempt_is_correct'].fillna(0.0)
        steps = scaler.transform(grp.values)
        n     = min(len(steps), max_len_raw)
        X[i, :n, :] = steps[:n]
        mask[i, :n]  = True
        y[i]         = int(label_map.get(lid, 0))
        ids.append(f'{lid}::{tc}')

    return X, y, mask, ids

X_train, y_train, mask_train, ids_train = build_tensors('train')
X_test,  y_test,  mask_test,  ids_test  = build_tensors('test')

print(f'Train tensor shape : X={X_train.shape}  y={y_train.shape}  mask={mask_train.shape}')
print(f'Test  tensor shape : X={X_test.shape}   y={y_test.shape}   mask={mask_test.shape}')
print(f'Train at_risk dist : {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Test  at_risk dist : {dict(zip(*np.unique(y_test,  return_counts=True)))}')

In [ ]:
# Tensor shape validation
assert X_train.ndim == 3,                        'X_train must be 3-D (N, T, F)'
assert X_train.shape[1] == max_len_raw,          'T dimension must equal max_len_raw'
assert X_train.shape[2] == N_FEATURES,           'F dimension must equal N_FEATURES'
assert X_train.shape[0] == mask_train.shape[0],  'N must match between X and mask'
assert mask_train.shape[1] == max_len_raw,        'mask T dimension must equal max_len_raw'
assert not np.isnan(X_train).any(),               'X_train contains NaN — fill or impute'
assert not np.isnan(X_test).any(),                'X_test contains NaN — fill or impute'
print('Tensor shape validation: PASS')

## 10. Write artifacts and sequence manifest

In [ ]:
# Sequence tensors
tensors_path = SEQ_DIR / 'sequence_tensors_v1.npz'
np.savez_compressed(
    tensors_path,
    X_train=X_train, y_train=y_train, mask_train=mask_train,
    X_test=X_test,   y_test=y_test,   mask_test=mask_test,
)
print(f'Tensors saved: {tensors_path}')

# Vocabulary
vocab_path = SEQ_DIR / 'vocabulary_v1.json'
vocab_doc  = {
    'schema_version':   SCHEMA_VERSION,
    'padding_token':    0,
    'event_type_vocab': vocab,
    'block_events_reserved': sorted(BLOCK_EVENT_TYPES),
    'note': 'Block event slots reserved for future phases — not collected in Phase 4',
}
vocab_path.write_text(json.dumps(vocab_doc, indent=2))
print(f'Vocabulary saved: {vocab_path}')

# Scaler parameters
scaler_path = SEQ_DIR / 'scaler_v1.json'
scaler_doc  = {
    'schema_version': SCHEMA_VERSION,
    'feature_names':  NUMERIC_FEATURES,
    'mean_':          scaler.mean_.tolist(),
    'scale_':         scaler.scale_.tolist(),
    'n_samples_seen': int(scaler.n_samples_seen_),
    'fit_split':      'train',
}
scaler_path.write_text(json.dumps(scaler_doc, indent=2))
print(f'Scaler saved: {scaler_path}')

In [ ]:
# Sequence manifest — the authoritative record for this M2 run
manifest = {
    'schema_version':           SCHEMA_VERSION,
    'created_at_utc':           datetime.now(timezone.utc).isoformat(),
    'phase3_source_sha':        PHASE3_SOURCE_SHA,
    'input_files': {
        'sequence_csv':  str(seq_path),
        'sequence_sha':  sha256_file(seq_path),
        'attempt_csv':   str(attempt_path),
        'attempt_sha':   sha256_file(attempt_path),
        'outcome_csv':   str(outcome_path),
        'outcome_sha':   sha256_file(outcome_path),
    },
    'parameters': {
        'at_risk_threshold':      AT_RISK_THRESHOLD,
        'dedup_window_sec':       DEDUP_WINDOW_SEC,
        'max_seq_len_percentile': MAX_SEQ_LEN_PERCENTILE,
        'max_seq_len':            max_len_raw,
        'n_features':             N_FEATURES,
        'feature_names':          NUMERIC_FEATURES,
        'random_state':           RANDOM_STATE,
        'test_size':              TEST_SIZE,
    },
    'dataset_stats': {
        'raw_events':             int(len(seq_df)),
        'dropped_as_duplicate':   int(n_dropped),
        'canonical_events':       int(len(canonical_df)),
        'pre_cutoff_events':      int(len(pre_cutoff_df)),
        'total_learners':         int(len(learner_labels)),
        'eligible_learners':      int(len(eligible)),
        'train_learners':         int(len(train_learners)),
        'test_learners':          int(len(test_learners)),
        'thesis_eligible_labels': int(learner_labels['thesis_eligible'].sum()),
        'pilot_only_labels':      int(learner_labels['pilot_only'].sum()),
        'train_shape':            list(X_train.shape),
        'test_shape':             list(X_test.shape),
    },
    'data_warning': (
        'TECHNICAL VALIDATION ONLY — generated from mock dataset. '
        'Not suitable for research conclusions. '
        f'Final thesis requires >=60 participants.'
    ),
    'artifacts': {
        'canonical_events':   str(canonical_path),
        'sequence_index':     str(seq_index_path),
        'split_assignments':  str(ledger_path),
        'sequence_tensors':   str(tensors_path),
        'vocabulary':         str(vocab_path),
        'scaler':             str(scaler_path),
    },
}

manifest_path = SEQ_DIR / 'sequence_manifest_v1.json'
manifest_path.write_text(json.dumps(manifest, indent=2, default=str))
print(f'Manifest saved: {manifest_path}')
print(json.dumps(manifest['dataset_stats'], indent=2))

## 11. Validation summary

In [ ]:
checks = []

def chk(name: str, passed: bool, detail: str = '') -> None:
    checks.append({'check': name, 'result': 'PASS' if passed else 'FAIL', 'detail': detail})

chk('Snapshot files loaded',
    all(p is not None for p in [seq_path, attempt_path, outcome_path]))

chk('Required columns present (all snapshots)', True,  # would have raised above if not
    'sequence / attempt / outcome')

chk('Event deduplication',
    n_dropped >= 0,
    f'{n_dropped} client-side duplicates dropped')

chk('Block events absent (Phase 4 text-mode only)',
    block_found == 0,
    f'{block_found} block events found')

chk('Cutoff timestamps computed',
    len(cutoff_df) > 0,
    f'{len(cutoff_df)} learner×task cutoffs')

chk('Anti-leakage (no blacklisted columns in features)',
    len(leaked) == 0,
    'blacklisted: ' + (', '.join(leaked) if leaked else 'none'))

chk('Split: no learner overlap',
    len(overlap) == 0,
    f'overlap={len(overlap)}')

chk('Tensor shape (3-D)',
    X_train.ndim == 3 and X_test.ndim == 3,
    f'train={X_train.shape}  test={X_test.shape}')

chk('No NaN in tensors',
    not np.isnan(X_train).any() and not np.isnan(X_test).any())

chk('Mask shape matches tensor',
    mask_train.shape == X_train.shape[:2] and mask_test.shape == X_test.shape[:2])

chk('Split ledger saved',
    ledger_path.exists())

chk('Manifest saved',
    manifest_path.exists())

chk('Label validity documented',
    'label_source' in split_ledger.columns,
    f'thesis_eligible={learner_labels["thesis_eligible"].sum()}  '
    f'pilot_only={learner_labels["pilot_only"].sum()}')

result_df = pd.DataFrame(checks)
n_fail    = (result_df['result'] == 'FAIL').sum()

print('\n── M2 Validation Summary ─────────────────────────────────────────')
print(result_df.to_string(index=False))
print(f'\n{len(checks) - n_fail}/{len(checks)} checks passed')

if n_fail > 0:
    raise RuntimeError(f'M2 validation FAILED — {n_fail} check(s) did not pass. See table above.')

print('\n✅ M2 COMPLETE — sequence dataset artifacts ready for M3 (TAG) and M4 (LSTM).')
print('\n⚠️  TECHNICAL VALIDATION ONLY — not suitable for research conclusions.')
print(f'   label_source=auto_generated / label_validity=pilot_only for all {len(eligible)} learners.')